# APDL Preview for `bcc`

Generate APDL command text for the `bcc` unit cell and inspect it before pasting it into MAPDL/APDL.

In [18]:
%pip install -e .

from pathlib import Path
import sys

%load_ext autoreload
%autoreload 2

project_root = Path.cwd().resolve()

src = project_root / "src"

for p in [str(src), str(project_root)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("project_root:", project_root)
print("src:", src)
print("sys.path[0:3]:", sys.path[:3])

Obtaining file:///C:/Users/USER/Documents/parametric_lattice
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for parametric-lattice (pyproject.toml): started
  Building editable for parametric-lattice (pyproject.toml): finished with status 'done'
  Created wheel for parametric-lattice: filename=parametric_lattice-0.1.0-0.editable-py3-none-any.whl size=4676 sha256=edea98ab3c350dc436ed644736c0cd64ff2d4aeb43faca68dd294ee691878122
  Stored in directory: C:\Users\USER\AppData\Local\Temp\pip-ephem-wheel-cache-rvt37rq7\wheels\db\36\3

In [ ]:
from apdl_preview import build_sim_case


sim_case = build_sim_case(
    "sc",
    element_type="BEAM188",
    sim_type="xx",  # static compression along x
    size_xyz=(10.0, 10.0, 10.0),
    radius=1.0,
    e_mod=210000.0,
    nu=0.3,
    density=7.85e-9,
    max_element_size=0.3,
    strain=0.01,
    n_substeps=1,
    joint_area_factor=4,
    joint_length_factor=2,
    joint_bending_factor=10000,
    joint_torsion_factor=10000,
    kappa=0.85
)

print(sim_case.to_string())

BEAM188__Radius:1.000000__Kappa:0.850000__JArea:4.000000__JLength:2.000000__JBend:10000.000000__JTors:10000.000000__Cell:sc__Size:10.000000,10.000000,10.000000__MaxElementSize:0.300000__E:210000.000000__Nu:0.300000__Density:0.000000__SimType:xx__Strain:0.010000__Substeps:1


In [61]:
from custom_io.apdl_io import generate_apdl_script
from pipeline import build_pipeline
from postprocess.pipeline import postprocess_commands

apdl_cmd = generate_apdl_script(
    build_pipeline(sim_case)
    + ("/POST1",)
    + postprocess_commands(
        sim_case=sim_case,
        needed={"boundary_traction": 9},
    )
)
print(apdl_cmd)

/CLEAR,START
/UNITS,MPA
/PREP7
! Define beam element type and material properties
ET,1,188
KEYOPT,1,3,3
KEYOPT,1,15,0
! ============================================================
! GEOMETRY DEFINITION
! ============================================================
! --- Create keypoints
K,1,-5,-5,-5                    ! 0: n 0.0 0.0 0.0
K,2,5,-5,-5                     ! 1: n 1.0 0.0 0.0
K,3,-5,5,-5                     ! 2: n 0.0 1.0 0.0
K,4,5,5,-5                      ! 3: n 1.0 1.0 0.0
K,5,-5,-5,5                     ! 4: n 0.0 0.0 1.0
K,6,5,-5,5                      ! 5: n 1.0 0.0 1.0
K,7,-5,5,5                      ! 6: n 0.0 1.0 1.0
K,8,5,5,5                       ! 7: n 1.0 1.0 1.0
! --- Create beam orientation keypoints from edge normal vectors
K,9,-5,-5,-4                    !  0: e_n [0. 0. 1.]
K,10,-5,-4,-5                   !  0: e_b [0. 1. 0.]
K,11,-5,-5,-6                   !  1: e_n [ 0.  0. -1.]
K,12,-4,-5,-5                   !  1: e_b [ 1. -0.  0.]
K,13,-5,-4,-5       

In [21]:
from custom_io.excel_io import run_cases


# run_cases((sim_case,))